# 코랩 음식 사진 배경 교체 검증

운영과 같은 안전 모드로 실행합니다. 기본 탐지기는 학습한 음식 전용 YOLO11n `models/best.pt`입니다. YOLO11n 탐지 실패 시 중앙 사각형을 사용하지 않으며, `food_detection_failed` 보고서를 남기고 광고 이미지를 만들지 않습니다.

In [ ]:
from google.colab import drive, files
from pathlib import Path
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/final_1_team/apps/api/food-image-cleanup-pipeline')
assert (PROJECT_ROOT / 'configs/pipeline.yaml').is_file(), f'프로젝트를 찾을 수 없습니다: {PROJECT_ROOT}'
%cd $PROJECT_ROOT

In [ ]:
import os, subprocess, sys
PACKAGE_DIR = Path('/content/food-image-cleanup-packages')
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(PACKAGE_DIR), '--prefer-binary', '-r', 'requirements-colab.txt'], check=True)
RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV['PYTHONPATH'] = str(PACKAGE_DIR) + os.pathsep + RUNTIME_ENV.get('PYTHONPATH', '')
print('파이프라인 의존성 설치 완료')

In [ ]:
# GroundingDINO도 Drive의 models/grounding-dino에 스냅샷으로 저장해 다음 실행부터 재사용합니다.
subprocess.run([sys.executable, '-m', 'scripts.download_models', '--models', 'yolo', 'sam2', 'big-lama', 'openclip', 'birefnet', 'sana', 'grounding-dino', 'hq-sam'], check=True, env=RUNTIME_ENV)
DETECTOR_PROFILE = 'food_specialized'  # 비교가 필요하면 coco_yolo11n으로 변경합니다.
DETECTOR_WEIGHTS = PROJECT_ROOT / 'models/best.pt' if DETECTOR_PROFILE == 'food_specialized' else PROJECT_ROOT / 'models/yolo11n.pt'
ANGLE_WEIGHTS = PROJECT_ROOT / 'models/efficientnet_best.pt'
SAM_SMALL_WEIGHTS = PROJECT_ROOT / 'models/sam2.1_s.pt'
GROUNDING_DINO_DIR = PROJECT_ROOT / 'models/grounding-dino'
HQ_SAM_DIR = PROJECT_ROOT / 'models/hq-sam'
assert SAM_SMALL_WEIGHTS.is_file(), f'SAM 2.1 Small 가중치가 없습니다: {SAM_SMALL_WEIGHTS}'
assert DETECTOR_WEIGHTS.is_file(), f'선택한 탐지기 가중치가 없습니다: {DETECTOR_WEIGHTS}'
assert ANGLE_WEIGHTS.is_file(), f'EfficientNet-B0 각도 분류 가중치가 없습니다: {ANGLE_WEIGHTS}'
assert (GROUNDING_DINO_DIR / 'config.json').is_file(), f'GroundingDINO 스냅샷이 없습니다: {GROUNDING_DINO_DIR}'
assert (HQ_SAM_DIR / 'config.json').is_file(), f'HQ-SAM snapshot is missing: {HQ_SAM_DIR}'
print(f'HQ-SAM local snapshot: {HQ_SAM_DIR}')
print(f'탐지 프로필: {DETECTOR_PROFILE}, 가중치: {DETECTOR_WEIGHTS}')
print(f'GroundingDINO 로컬 스냅샷: {GROUNDING_DINO_DIR}')
print(f'각도 분류기: {ANGLE_WEIGHTS}')
print(f'전경 탐지 순서: GroundingDINO 우선 → YOLO 보완, SAM: {SAM_SMALL_WEIGHTS.name}')

In [ ]:
import ipywidgets as widgets
from IPython.display import display

business_type_widget = widgets.Dropdown(
    options=['cafe', 'restaurant', 'bakery', 'dessert shop', 'bar', 'food truck', 'premium dining', 'home meal replacement'],
    value='cafe',
    description='Business',
)
custom_business_type_widget = widgets.Text(value='', placeholder='Optional custom business type', description='Custom')
desired_mood_widget = widgets.Dropdown(
    options=['cozy and warm', 'modern and clean', 'premium and elegant', 'bright and fresh', 'rustic and natural', 'playful and colorful'],
    value='cozy and warm',
    description='Mood',
)
custom_desired_mood_widget = widgets.Text(value='', placeholder='Optional custom mood', description='Custom')
composition_mode_widget = widgets.Dropdown(
    options=[('Food only on generated plate', 'generated_plate'), ('Keep original plate and food', 'preserve_original_plate')],
    value='generated_plate',
    description='Foreground',
)
display(business_type_widget, custom_business_type_widget, desired_mood_widget, custom_desired_mood_widget, composition_mode_widget)


In [ ]:
import json, shutil
from PIL import Image
from IPython.display import display
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one food image.'
source_path = Path(next(iter(uploaded)))
input_path = Path('data/input') / f'example{source_path.suffix.lower()}'
input_path.parent.mkdir(parents=True, exist_ok=True)
shutil.move(str(source_path), input_path)
business_type = custom_business_type_widget.value.strip() or business_type_widget.value
desired_mood = custom_desired_mood_widget.value.strip() or desired_mood_widget.value
composition_mode = composition_mode_widget.value
metadata = {
    'business_type': business_type,
    'desired_mood': desired_mood,
    'food_category': 'food',
    'foreground_position': 'center_lower',
    'light_direction': 'left',
    'composition_mode': composition_mode,
    'require_food_visible_mask': composition_mode != 'generated_plate',
}
metadata_path = Path('data/input/example_metadata.json')
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(metadata, ensure_ascii=False, indent=2))
display(Image.open(input_path))


In [ ]:
# --diagnostic-center-fallback을 넣지 않습니다. 탐지 실패는 운영과 같이 안전하게 차단됩니다.
command = [sys.executable, '-m', 'scripts.run_background_replacement', '--input', str(input_path), '--metadata', str(metadata_path), '--enable-matting', '--enable-background-generator', '--detector-profile', DETECTOR_PROFILE]
result = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True, env=RUNTIME_ENV)
print(result.stdout)
if result.stderr: print(result.stderr)
print('종료 코드:', result.returncode)

In [ ]:
from IPython.display import Image as DisplayImage
report_path = Path('data/reports') / f'{input_path.stem}_background_replacement_report.json'
assert report_path.is_file(), f'보고서가 없습니다: {report_path}'
report = json.loads(report_path.read_text(encoding='utf-8'))
detector_stage = report.get('stages', {}).get('step_2_yolo_detection', {})
angle_stage = report.get('stages', {}).get('step_7_camera_angle_classification', {})
assert detector_stage.get('profile') == DETECTOR_PROFILE, f'탐지 프로필 불일치: {detector_stage}'
if DETECTOR_PROFILE == 'food_specialized':
    assert detector_stage.get('model', '').replace('\\', '/').endswith('models/best.pt'), detector_stage
assert angle_stage.get('model', '').replace('\\', '/').endswith('models/efficientnet_best.pt'), angle_stage
assert angle_stage.get('status') in {'completed', 'low_confidence'}, angle_stage
assert angle_stage.get('label') in {'top', '45'}, angle_stage
print(json.dumps({'status':report.get('status'), 'reason':report.get('reason'), 'detector':detector_stage, 'camera_angle':angle_stage, 'debug_artifacts':report.get('debug_artifacts'), 'validation':report.get('stages',{}).get('step_13_foreground_validation')}, ensure_ascii=False, indent=2))
for name, artifact_path in report.get('debug_artifacts', {}).items():
    artifact = Path(artifact_path)
    if artifact.is_file():
        print(name, artifact)
        display(DisplayImage(filename=str(artifact)))

## 실험 산출물 자동 저장

실행 성공·차단·검증 실패와 관계없이 입력, 보고서, 결과와 디버그 산출물을 Drive의 시간별 실험 폴더에 보관합니다.

In [ ]:
from datetime import datetime

EXPERIMENT_DIR = (
    PROJECT_ROOT / 'data/experiments/background_replacement'
    / datetime.now().strftime('%Y%m%d_%H%M%S')
)
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=False)

def project_path(value):
    candidate = Path(value)
    return candidate if candidate.is_absolute() else PROJECT_ROOT / candidate

artifacts_to_save = {
    'input': input_path,
    'metadata': metadata_path,
    'report': report_path,
}
for name, value in report.get('debug_artifacts', {}).items():
    if value:
        artifacts_to_save[f'debug_{name}'] = project_path(value)
for key in ('output_path', 'foreground_path'):
    if report.get(key):
        artifacts_to_save[key] = project_path(report[key])

saved = {}
for name, source in artifacts_to_save.items():
    source = project_path(source)
    if source.is_file():
        target = EXPERIMENT_DIR / source.name
        shutil.copy2(source, target)
        saved[name] = str(target.relative_to(PROJECT_ROOT))

manifest = {
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'status': report.get('status', 'completed'),
    'reason': report.get('reason'),
    'saved_files': saved,
}
(EXPERIMENT_DIR / 'experiment_manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('실험 산출물 저장 완료:', EXPERIMENT_DIR)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
